In [ ]:
import json
import os
import glob
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, classification_report, precision_recall_curve
from catboost import CatBoostClassifier

# 📌 외부 파일에서 딥러닝 모델 클래스 임포트
from models import ChurnLSTM, ChurnCNN1D

# ==========================================
# 🚀 1. 디렉토리 구조 설정
# ==========================================
DATA_DIR = "./data/data/processed"
ML_MODEL_DIR = "./model/machine_learning"
DL_MODEL_DIR = "./model/deep_learning"
EVAL_DIR = "./test"

os.makedirs(EVAL_DIR, exist_ok=True)

print("=" * 50)
print("🚀 1. 데이터 로드 및 테스트셋 준비 (추론 및 앙상블 전용)")
print("=" * 50)

# ==========================================
# 🚀 2. 데이터 로드 및 테스트셋 분할
# ==========================================
csv_path = os.path.join(DATA_DIR, "customer_features_20181101_v1.csv")
if not os.path.exists(csv_path):
    csv_path = os.path.join(DATA_DIR, "train_20181101_v1.csv")

df_ml = pd.read_csv(csv_path)

seq_path = os.path.join(DATA_DIR, "converted_sequence_tensor.npy")
if not os.path.exists(seq_path):
    seq_path = os.path.join(DATA_DIR, "dl_sequence_tensor.npy")

X_seq = np.load(seq_path)
target_col = "churn" if "churn" in df_ml.columns else ("is_churn" if "is_churn" in df_ml.columns else df_ml.columns[-1])
y = df_ml[target_col].values

# 정형 데이터 피처 정제
drop_cols = [target_col, "userId", "page_sequence", "session_item_sequence", "padded_sequence"]
feature_cols = [col for col in df_ml.columns if col not in drop_cols]
X_tab = df_ml[feature_cols].copy()

for col in ["gender", "level", "last_page"]:
    if col in X_tab.columns:
        X_tab[col] = X_tab[col].astype("category")

for col in X_tab.columns:
    if X_tab[col].dtype == "object" and col not in ["gender", "level", "last_page"]:
        X_tab[col] = pd.to_numeric(X_tab[col], errors="coerce").fillna(0.0)

# 학습 시와 동일한 난수 고정(random_state=42)으로 테스트셋 추출
indices = np.arange(len(y))
_, test_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

X_tab_test = X_tab.iloc[test_idx]
X_seq_test = X_seq[test_idx]
y_test = y[test_idx]
y_true = np.array(y_test).ravel()

# DL 테스트 데이터로더 구성
test_dataset = TensorDataset(torch.tensor(X_seq_test, dtype=torch.long), torch.tensor(y_test, dtype=torch.float32))
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# 🚀 3. 모델 디렉토리 스캔 및 예측값 수집
# ==========================================
print("\n" + "=" * 50)
print("🚀 2. 저장된 모델 스캔 및 예측 수행")
print("=" * 50)

# 3.1 머신러닝 모델 예측값 수집 (.pkl 및 .cbm 파일 동시 지원)
ml_predictions = {}
ml_model_files = glob.glob(os.path.join(ML_MODEL_DIR, "*.pkl")) + glob.glob(os.path.join(ML_MODEL_DIR, "*.cbm"))

if not ml_model_files:
    print(f"⚠️ 경고: {ML_MODEL_DIR} 경로에 저장된 머신러닝 모델(.pkl, .cbm)이 없습니다.")

for model_path in ml_model_files:
    model_name = os.path.basename(model_path)
    
    # 파일 확장자에 따라 다르게 로드
    if model_name.endswith(".pkl"):
        model = joblib.load(model_path)
    elif model_name.endswith(".cbm"):
        model = CatBoostClassifier()
        model.load_model(model_path)
    else:
        raise ValueError(f"지원하지 않는 머신러닝 모델 형식입니다: {model_name}")
        
    preds = model.predict_proba(X_tab_test)[:, 1]
    ml_predictions[model_name] = preds
    print(f"📥 [ML 로드 완료] {model_name} (Test AUC: {roc_auc_score(y_true, preds):.4f})")

# 3.2 딥러닝 모델 예측값 수집
dl_predictions = {}
dl_model_files = glob.glob(os.path.join(DL_MODEL_DIR, "*.pth"))
if not dl_model_files:
    print(f"⚠️ 경고: {DL_MODEL_DIR} 경로에 저장된 딥러닝 모델 가중치(.pth)가 없습니다.")

for model_path in dl_model_files:
    model_name = os.path.basename(model_path)
    
    if "lstm" in model_name.lower():
        dl_model = ChurnLSTM(vocab_size=50, embedding_dim=32, hidden_dim=64).to(device)
    elif "cnn" in model_name.lower():
        dl_model = ChurnCNN1D(vocab_size=50, embedding_dim=32, num_filters=64).to(device)
    else:
        raise ValueError(f"지원하지 않는 딥러닝 모델 파일명입니다: {model_name}")
        
    dl_model.load_state_dict(torch.load(model_path, map_location=device))
    dl_model.eval()
    
    dl_preds_list = []
    with torch.no_grad():
        for seqs, _ in test_loader:
            outputs = torch.sigmoid(dl_model(seqs.to(device)))
            dl_preds_list.extend(outputs.cpu().numpy())
    
    preds = np.array(dl_preds_list).ravel()
    dl_predictions[model_name] = preds
    print(f"📥 [DL 로드 완료] {model_name} (Test AUC: {roc_auc_score(y_true, preds):.4f})")

# ==========================================
# 🚀 4. 모든 ML x DL 조합 중 최고 성능 탐색
# ==========================================
if not ml_predictions or not dl_predictions:
    raise ValueError("앙상블을 수행할 ML 모델 또는 DL 모델 파일이 부족합니다. 디렉토리를 확인해주세요.")

best_overall_auc = 0
best_combination = None
best_hybrid_preds = None
best_weights = (0.5, 0.5)

for ml_name, ml_preds in ml_predictions.items():
    for dl_name, dl_preds in dl_predictions.items():
        for w in np.linspace(0.5, 0.95, 10):
            temp_preds = (w * ml_preds) + ((1.0 - w) * dl_preds)
            score = roc_auc_score(y_true, temp_preds)
            if score > best_overall_auc:
                best_overall_auc = score
                best_combination = (ml_name, dl_name)
                best_hybrid_preds = temp_preds
                best_weights = (w, 1.0 - w)

print("\n" + "★" * 50)
print(f"🏆 최적 하이브리드 조합 발견!")
print(f" - 머신러닝 모델: {best_combination[0]} (가중치: {best_weights[0]:.2f})")
print(f" - 딥러닝 모델: {best_combination[1]} (가중치: {best_weights[1]:.2f})")
print(f" - 🏆 최종 하이브리드 Test ROC-AUC Score: {best_overall_auc:.4f}")
print("★" * 50)

🚀 1. 데이터 로드 및 테스트셋 준비 (추론 및 앙상블 전용)

🚀 2. 저장된 모델 스캔 및 예측 수행
📥 [ML 로드 완료] lgb_model.pkl (Test AUC: 0.7163)
📥 [ML 로드 완료] catboost_model.cbm (Test AUC: 0.7091)
📥 [DL 로드 완료] cnn1d_model_weights.pth (Test AUC: 0.8332)
📥 [DL 로드 완료] lstm_model_weights.pth (Test AUC: 0.8545)

★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★
🏆 최적 하이브리드 조합 발견!
 - 머신러닝 모델: lgb_model.pkl (가중치: 0.70)
 - 딥러닝 모델: lstm_model_weights.pth (가중치: 0.30)
 - 🏆 최종 하이브리드 Test ROC-AUC Score: 0.8910
★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★★

📌 재현율 우선 최적 임계값 (F2-Score 기준): 0.1958

              precision    recall  f1-score   support

           0       0.99      0.79      0.88      3421
           1       0.20      0.86      0.33       207

    accuracy                           0.80      3628
   macro avg       0.60      0.83      0.60      3628
weighted avg       0.94      0.80      0.85      3628


모든 추론, 최적화 및 결과 저장 과정이 ./test 디렉토리에 완료되었습니다!


In [ ]:

# ==========================================
# 🚀 5. 최적 임계값 적용 및 평가 리포트 출력
# ==========================================
precisions, recalls, thresholds = precision_recall_curve(y_true, best_hybrid_preds)

p = precisions[:-1]
r = recalls[:-1]
t = thresholds

# 2. F_beta Score 계산 (beta = 2로 설정하여 재현율에 가중치 부여)
beta = 2.0
f_beta_scores = (
    (1 + beta**2) * (p * r) / ((beta**2 * p) + r + 1e-10)
)

# 3. F_beta가 가장 높은 최적의 임계값 선택
best_idx = np.argmax(f_beta_scores)
best_threshold = t[best_idx]

print(f"\n📌 재현율 우선 최적 임계값 (F2-Score 기준): {best_threshold:.4f}\n")
optimal_preds = (best_hybrid_preds >= best_threshold).astype(int)
print(classification_report(y_true, optimal_preds))

# 결과 설정 및 예측값 저장 (./test 디렉토리)
hybrid_config = {
    "best_ml_model": best_combination[0],
    "best_dl_model": best_combination[1],
    "ml_weight": float(best_weights[0]),
    "dl_weight": float(best_weights[1]),
    "best_auc": float(best_overall_auc),
    "best_threshold": float(best_threshold)
}
with open(os.path.join(EVAL_DIR, "best_hybrid_config.json"), "w", encoding="utf-8") as f:
    json.dump(hybrid_config, f, ensure_ascii=False, indent=4)

np.save(os.path.join(EVAL_DIR, "best_hybrid_preds.npy"), best_hybrid_preds)
print("\n모든 추론, 최적화 및 결과 저장 과정이 ./test 디렉토리에 완료되었습니다!")